# **HW 1 of 2. Linear models: weights, regularization and what they predict**

### What is inside

This notebook is the practice for the lessons **"Linear regression"** and **"What does a regression predict?"**. We will go through the whole chain of questions about the weights of a linear model:

1. how to read the weights as a feature's contribution (bar plot, relative importance);
2. why a single point estimate of a weight is not enough — importance via cross-validation, `MEAN/SE` and the outlier-robust `MEDIAN` version;
3. how to diagnose a model by its residuals;
4. multicollinearity and VIF — where the weight goes when features correlate;
5. what a model predicts at all depending on the loss function (MSE → the mean, quantile loss → a quantile);
6. the meaning of the intercept $w_0$ and what regularization does to the weights (Ridge, Lasso, L0).

$$y = \sum_{i=1}^{n}x_i\beta_i + \beta_0 = \beta_0 + x_1\beta_1+....+x_n\beta_n + \varepsilon$$

### **A reminder: the limitations on applying the model**

In the theory we singled out 4+4 limitations. The first 4 are informal, "semantic" ones:
___
1. The features have to be meaningful — that is, you understand what is being multiplied by the weight.
2. The scales of the features are comparable with each other.
3. The coefficients of the model are stable — not always required, but on basic tasks it is a desirable property.
4. The dependencies between features must not destroy the model — under strong correlation of features the coefficients of a linear model can become unstable.
___
The second four set the context that lets us say whether the estimate of the weights is mathematically correct:

1. **Linearity** — the conditional mean of the target variable $y$ is linearly related to the features $x_i$: $$\mathbb{E}[Y|X] = \beta_0 + \sum_i x_i\beta_i$$

   Under nonlinearity you can try data transformations to fix the situation. As a reminder, the standard ones are:

      - **For continuous features:** taking logarithms, taking a root, the `z-score` transformation, `StandardScaler`, `MinMaxScaler` and [others](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.preprocessing).

      - **For categorical features:** One-Hot encoding. To avoid a linear dependence one usually drops the column of one of the categories — but only if ALL the possible categories are present in the dataset (otherwise rows will remain where every OHE column equals 0).

      - The approaches can also be combined — for example, splitting a continuous feature into quantiles ([`pd.cut`, `pd.qcut`](https://pbpython.com/pandas-qcut-cut.html)).

2. **Independence of the objects of the sample** — the objects must enter the sample independently of one another.
3. **Absence of multicollinearity of the features** — the correlation of the features with each other is weak or absent.
4. **Normality of the residuals and homoscedasticity of the errors** — two different but important assumptions about the unpredicted part of the model. Normality means that the errors $\varepsilon = y - \hat{y}$ are approximately normally distributed. Homoscedasticity means that the variance of those errors is approximately the same at different values of the features or the predictions: $Var(\varepsilon|X)=\sigma^2$.

In this homework we will look at how various artefacts of the real world (correlation, unevenness of the data) affect the stability and the weights of a regression, and we will also get a hands-on feel for every block of theory we have studied. Off we go!

To start with, let us simply train a model.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.datasets import fetch_california_housing

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import mean_absolute_error, r2_score

Let us fix the randomness.

In [ ]:
RANDOM_STATE = 42

As the dataset we use [California Housing](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset), built into sklearn. The target variable `y` is the median house value in hundreds of thousands of US dollars.

In [ ]:
data = fetch_california_housing(as_frame=True)

X = data.data
y = data.target

X.head()

Let us split the data into a training and a test sample, not forgetting the scaling.

**Q1: Why is scaling applied to linear models?**

`Select all correct statements in the trainer`

**Q2: How does `StandardScaler()` scaling behave?**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=RANDOM_STATE, test_size=0.25)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Let us train a linear regression and compare its prediction with a baseline one — predicting the training-sample mean for every house.

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

predictions = lr.predict(X_test_scaled)
base = np.array([y_train.mean()]*len(X_test)) # predict the mean

print('Baseline algorithm quality (MAE): ', mean_absolute_error(y_test, base))
print('Linear regression quality (MAE): ', mean_absolute_error(y_test, predictions))

In [ ]:
print('Baseline algorithm quality (R2): ', r2_score(y_test, base))
print('Linear regression quality (R2): ', r2_score(y_test, predictions))

Not great, but not terrible either! The regression is built, it is better than the baseline, but still not ideal — this has to do with the nature of the data. 

## Block 2. Weights as a feature's contribution

After training a linear regression we have the weights — the strength of every feature's contribution to the prediction. They are stored in the `lr.coef_` attribute.

In [ ]:
labels = X_train.columns # feature names
values = lr.coef_ # feature weight values

Let us look at the values in a table.

In [ ]:
weights_data = pd.DataFrame(values, index=labels, columns=['weight'])

weights_data

**Q3.** Which feature is the most significant in the model by the absolute value of its weight?

The weights can be both positive and negative — the sign reflects the direction of the relation. One way to present them visually is a bar plot. The first version keeps the sign:

In [ ]:
plt.figure(figsize=(12, 4))

bar = plt.bar(height=lr.coef_, x=labels)
plt.bar_label(bar, padding=-13, color='black')

plt.title('Linear regression feature importance based on the weights');

The second uses the absolute values:

In [ ]:
plt.figure(figsize=(12, 4))

bar = plt.bar(height=np.abs(values), x=labels)
plt.bar_label(bar, color='black')

plt.title('Linear regression feature importance based on the weights (absolute value)');

## Block 2.1. Relative feature importance

A more convenient representation of a feature's contribution can be computed head-on — by normalizing every coefficient by the sum of the absolute values of all the weights. We get the share of every feature in the total "weight" of the model — convenient when you need to convey relative significance.

In [ ]:
weights_data2 = pd.DataFrame([labels, values]).T
weights_data2.columns = ['feature', 'feature_weight']

weights_sum2 = sum(abs(weights_data2['feature_weight']))

weights_data2['feature_weight_normalized'] = weights_data2['feature_weight'].apply(lambda x: round(x/weights_sum2*100, 2))

In [ ]:
weights_data2.sort_values(by='feature_weight_normalized', ascending=False)

**Q4.** Analyse the relative contributions of the features to the model. Select the correct statements

## Block 3. Importance via cross-validation: `MEAN(β)/SE(β)`

In the lesson's theory we computed a feature's importance as $|MEAN(\beta)/SE(\beta)|$ over several training iterations (the coffee-shop example). The weights from Block 2 are exactly **one** such iteration, on one particular `train/test` split. How reliable is it? Let us check by repeating the training on different folds — the same way it will be done in the second homework for logistic regression, only here we do it first.

Let us train the model on 9 different subsamples (3 `KFold` objects with different `random_state`, 3 folds each) and collect the weights.

In [ ]:
coefs = []

for rs in [12, 7, 13]:
    kf = KFold(n_splits=3, shuffle=True, random_state=rs)
    for train_idx, val_idx in kf.split(X_train_scaled):
        model = LinearRegression()
        model.fit(X_train_scaled[train_idx], y_train.values[train_idx])
        coefs.append(model.coef_)

coefs = np.array(coefs) # (9, n_features)
coefs.shape

Let us compute `MEAN(β)`, `SE(β)` (the ordinary standard deviation over the iterations — as in the coffee-shop example in the theory) and the importance $|\frac{MEAN}{SE}|$.

In [ ]:
mean_ = coefs.mean(axis=0)
se_ = coefs.std(axis=0, ddof=1)
importance_mean = np.abs(mean_ / se_)

cv_report = pd.DataFrame({
    'feature': labels,
    'point_estimate': values,
    'MEAN(beta)_cv': mean_,
    'ABS(MEAN(beta)_cv)': np.abs(mean_),
    'SE(beta)_cv': se_,
    'importance_mean=|MEAN/SE|': importance_mean,
})

cv_report.sort_values('importance_mean=|MEAN/SE|', ascending=False)

**Q5.** Which feature is the most important by the $|MEAN/SE|$ estimate? Does it coincide with the largest weight by absolute value from Block 2 (`Latitude`)?

**Q6.** Which pair of features has the largest **absolute AND mean AND se**?

In [ ]:
cv_report.sort_values(['ABS(MEAN(beta)_cv)', 'SE(beta)_cv'], ascending=False)

### The robust version: the median instead of the mean

In the lesson "What does a regression predict?" we found out that the mean is sensitive to outliers while the median is robust to them (the MSE optimum is the mean, the MAE optimum is the median). The same principle works here: if at least one of the 9 CV iterations happens to land on a "noisy" fold, `MEAN` and `SE` will see it and shift, while the median will not.

Let us compute `MEDIAN(β)` and the robust analogue of `SE` — **MAD** (median absolute deviation), scaled by $1.4826$ (for a normal distribution this makes MAD comparable in scale with the standard deviation).

In [ ]:
median_ = np.median(coefs, axis=0)
mad_ = np.median(np.abs(coefs - median_), axis=0) * 1.4826
importance_median = np.abs(median_ / mad_)

cv_report['MEDIAN(beta)_cv'] = median_
cv_report['robust_SE(MAD*1.4826)'] = mad_
cv_report['importance_median=|MEDIAN/robustSE|'] = importance_median

cv_report.sort_values('importance_median=|MEDIAN/robustSE|', ascending=False) #.head(5)

In [ ]:
cv_report.sort_values('importance_mean=|MEAN/SE|', ascending=False)

**Q7.** Compare the two rankings — by $|MEAN/SE|$ and by $|MEDIAN/\text{robust SE}|$. Do the first 5 features coincide?

* **Pay attention to how the importance weight of AveOccup changes. Extract the coefficients of the feature.**

In [ ]:
coefs[:, 5] # AveOccup weighs

## Block 4. Diagnostics by the residuals

The lesson's theory says: after training you have to check the residuals for normality and homoscedasticity. Let us look at that in practice — first the prediction against the actual value.

In [ ]:
plt.scatter(predictions, y_test)
plt.xlabel('Prediction')
plt.ylabel('Actual')
plt.title('Scatter plot of the predicted values against the real data');

At this step you most likely already see the reason, but let us go into detail. We will split the objects by the size of the residual into groups (`low` / `middle` / `high` — by the quartiles of the absolute residual) and look at where they concentrate relative to one of the features.

In [ ]:
residuals = pd.Series(abs(predictions - y_test))

q1 = residuals.quantile(0.25)
q2 = residuals.quantile(0.75)

def get_resid_class(x, q1=q1, q2=q2):
    if x <= q1:
        return 'low'
    elif x <= q2:
        return 'middle'
    else:
        return 'high'

residuals_class = residuals.apply(get_resid_class)

In [ ]:
sns.scatterplot(x=X_test['MedInc'], y=y_test, hue=residuals_class);
plt.legend();
plt.title('Residual class based on the MedInc feature and the target variable.', 
          pad=15);

**Q8.** Analyse the plot. Which contradiction (or contradictions) gets in the way of the linearity of the relation, and where is it observed? Select the correct statements.

## Block 5. Multicollinearity

In Block 3 we saw that `Latitude` is the leader by the point estimate but not by robustness to resampling, and from the theory of the lessons we know that such behaviour can be a consequence of multicollinearity. Let us check whether that is so.

**Q9\*.** Compute the correlation between `Latitude` and `Longitude` on `X_train`, and then the VIF (variance inflation factor) for all the features.

In the answer field enter the value of the correlation rounded to two decimal places.

**Important:** `variance_inflation_factor` from `statsmodels` requires an explicitly added constant (`sm.add_constant`) — without it the numbers will be strongly inflated and meaningless.

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

print('Latitude/Longitude correlation:', X_train['Latitude'].corr(X_train['Longitude']))

X_train_c = sm.add_constant(X_train)
vif_data = pd.DataFrame()
vif_data['feature'] = X_train_c.columns
vif_data['VIF'] = [variance_inflation_factor(X_train_c.values, i) for i in range(X_train_c.shape[1])]
vif_data.round(2)

**Q10**. VIF and $R^2$

VIF and the $R^2$ of the auxiliary regression are one and the same number in two scales:

$$VIF_j = \frac{1}{1-R_j^2}, \qquad R_j^2 = 1 - \frac{1}{VIF_j},$$

where $R_j^2$ is the quality of the regression of the feature $x_j$ on all the other features (it is exactly this regression that is computed inside variance_inflation_factor). Recover $R_j^2$ from the VIFs you have already computed (using the formula only), and then check yourself: train a LinearRegression for Latitude and Longitude on the other features directly and compare .score() with what you got from the formula.

As the answer give the sum $R^2_{Latitude} + R^2_{Longitude}$ rounded to two decimal places. 

In [ ]:

# Recovering R^2 from the formula — without a single new model
vif_data['R2_from_VIF'] = 1 - 1 / vif_data['VIF']
vif_data.round(4)

# A direct regression check: Latitude ~ the other features
X_other = X_train.drop(columns=['Latitude'])
y_target = X_train['Latitude']

aux_model = LinearRegression().fit(X_other, y_target)
r2_direct = aux_model.score(X_other, y_target)

vif_latitude = vif_data.loc[vif_data['feature'] == 'Latitude', 'VIF'].values[0]
r2_from_vif = 1 - 1/vif_latitude

print('R^2 directly (LinearRegression.score)      :', round(r2_direct, 4))
print('R^2 recovered from the formula 1 - 1/VIF   :', round(r2_from_vif, 4))

In [ ]:
0.8907 + 0.8866

In [ ]:
# A direct regression check: Latitude ~ the other features
X_other = X_train.drop(columns=['Longitude'])
y_target = X_train['Longitude']

aux_model = LinearRegression().fit(X_other, y_target)
r2_direct = aux_model.score(X_other, y_target)

vif_longtitide = vif_data.loc[vif_data['feature'] == 'Longitude', 'VIF'].values[0]
r2_from_vif = 1 - 1/vif_longtitide

print('R^2 directly (LinearRegression.score)      :', round(r2_direct, 4))
print('R^2 recovered from the formula 1 - 1/VIF   :', round(r2_from_vif, 4))

## Block 6. What does a regression predict? Checking in practice

The lesson's theory covers this: an MSE model learns to predict the **conditional mean** $\mathbb{E}[Y\mid X=x]$, while a model with the pinball (quantile) loss for $\tau$ predicts the **$\tau$-quantile** of the conditional distribution. At $\tau=0.5$ this is the median — the same functional that MAE elicits.

Our `target` (the median house value) is right-skewed (skew $\approx 0.98$: mean $\approx 2.07$, median $\approx 1.80$) — that is, the mean and the median differ noticeably. So if the theory is right, models with different loss functions should predict differently on average.

**Q11.** Train on `X_train_scaled`, `y_train`:
1. `LinearRegression` (the reference MSE optimum → it should aim at the mean).
2. `sklearn.linear_model.QuantileRegressor(quantile=0.5, alpha=0, solver='highs')` (the median).
3. `QuantileRegressor(quantile=0.1, ...)` and `QuantileRegressor(quantile=0.9, ...)`.

Compare the mean predictions of every model on `X_test_scaled` with `y_test.mean()` and `y_test.median()`.
In the answer select the model that predicts the largest mean value. 

In [ ]:
from sklearn.linear_model import QuantileRegressor

pred_mse = lr.predict(X_test_scaled)

qr50 = QuantileRegressor(quantile=0.5, alpha=0, solver='highs').fit(X_train_scaled, y_train)
qr10 = QuantileRegressor(quantile=0.1, alpha=0, solver='highs').fit(X_train_scaled, y_train)
qr90 = QuantileRegressor(quantile=0.9, alpha=0, solver='highs').fit(X_train_scaled, y_train)

pred_median = qr50.predict(X_test_scaled)
pred_q10 = qr10.predict(X_test_scaled)
pred_q90 = qr90.predict(X_test_scaled)

mae_mse_model = np.mean(np.abs(pred_mse - y_test.values))
mae_median_model = np.mean(np.abs(pred_median - y_test.values))
print('MAE of the MSE model:     ', round(mae_mse_model, 4))
print('MAE of the median model: ', round(mae_median_model, 4))

In [ ]:
print('y_train.mean():  ', round(y_train.mean(), 3),
    ' | y_train.median():', round(y_train.median(), 3), 
    ' | y_train.quantile(0.5) :', round(y_train.quantile(0.5), 3),
    ' | y_train.quantile(0.1) :', round(y_train.quantile(0.1), 3),
    ' | y_train.quantile(0.9) :', round(y_train.quantile(0.9), 3),
    )

print('y_test.mean():  ', round(y_test.mean(), 3),
    ' | y_test.median():', round(y_test.median(), 3), 
    ' | y_test.quantile(0.5) :', round(y_test.quantile(0.5), 3),
    ' | y_test.quantile(0.1) :', round(y_test.quantile(0.1), 3),
    ' | y_test.quantile(0.9) :', round(y_test.quantile(0.9), 3),
    )


print('MSE model, mean prediction:       ', round(pred_mse.mean(), 3))

print('Quantile tau=0.5, mean prediction:', round(pred_median.mean(), 3))
print('Quantile tau=0.1, mean prediction:', round(pred_q10.mean(), 3))
print('Quantile tau=0.9, mean prediction:', round(pred_q90.mean(), 3))


## Block 7. The meaning of $\beta_0$ and what regularization does

The lesson's theory derives this: $\beta_0$ is **not** the expectation of the target variable "in general", but the model's prediction when all the features equal their means (or zero — if standardization was applied).

We standardized the features at the very beginning — so for our model, provided the training went well (with small errors, i.e. residuals), the following should hold:

$$\beta_0 = lr.intercept\_ \approx \bar y_{train}.$$

Let us check.

In [ ]:
print('OLS (MSE): lr.intercept_ =', round(lr.intercept_, 4), 
      ' | y_train.mean() =', round(y_train.mean(), 4),
      ' | diff =', round(lr.intercept_ - y_train.mean(), 6))
print()

for tau, model, name in [(0.1, qr10, 'qr10'), (0.5, qr50, 'qr50'), (0.9, qr90, 'qr90')]:
    q_train = y_train.quantile(tau)
    print(f'{name} (tau={tau}): intercept_ = {model.intercept_:.4f} | y_train.quantile({tau}) = {q_train:.4f} | diff = {model.intercept_ - q_train:.4f}')

For OLS the equality `lr.intercept_` $\approx$ `y_train.mean()` holds with sufficient accuracy. But for other kinds of regression we see significant differences. Let us check how our models err. 

In [ ]:
# For OLS the first-order condition is "the sum of train residuals equals zero" (a value).
# For quantile regression the first-order condition is different — "the share of train residuals below zero equals tau" (a rank, not a value).

for tau, model, name in [('none', lr, 'OLS'), (0.1, qr10, 'qr10'), (0.5, qr50, 'qr50'), (0.9, qr90, 'qr90')]:
    resid_train = y_train.values - model.predict(X_train_scaled)
    frac_below = (resid_train < 0).mean()
    total_sum = (resid_train).sum()
    print(f'{name}: share of train residuals < 0 = {frac_below:.4f}  (expected tau={tau}), total_sum={total_sum:.4f}')

**What happened?**

1. OLS.
    - total_sum=0.0000 — this is the identity that gives $\beta_0=\bar y_{train}$ (the least-squares equation).
    - the share of residuals <0 is 0.5858: for 58.6% of the houses the model overpriced, and only for 41.4% underpriced. OLS does not care about this — the optimality condition of that function is the minimization of squared errors. We saw that the target is right-skewed (Block 6), so it pays an MSE model to overprice slightly for the bulk of inexpensive houses, so as not to pay the quadratic penalty for underestimating the few expensive ones.


2. The quantile models.
    - The share of residuals <0 is close to $\tau$ (0.0996, 0.4997, 0.8998) — this is their real guarantee, proved in the lesson's theory. We said that it pays a model to predict the $\tau$-quantile. Then by definition, if $\hat y$ is the $\tau$% quantile ($\tau=0.1$), only $\tau$% of the values of $Y$ lie below that threshold, and 90% above it. That is:

    $$P(Y < \hat y) = \tau, \qquad P(Y > \hat y) = 1-\tau$$

    A residual is resid = y - pred. So:
    - resid < 0 $\iff$ y < pred $\iff$ the object is below the predicted quantile
    - resid > 0 $\iff$ y > pred $\iff$ the object is above the predicted quantile


    Hence $P(resid<0) = P(y<pred) = \tau$

    - the high values of total_sum follow from the fact that the pinball loss does not penalize the size of the error.


Generalizing: on train, every loss function guarantees exactly the kind of residual balance and the interpretation that follows from its optimal solution. 

**Bonus.** What will happen to the OLS equality if you train the model on **non**-standardized features (the raw `X_train`, without `StandardScaler`)? Will $w_0 \approx \bar y_{train}$ still hold? Check it in practice and explain the result, relying on the formula $w_0 = \bar y - \sum_i w_i \bar x_i$ from the theory.

This task is not graded and can be skipped =) 

In [ ]:
lr_raw = LinearRegression()
lr_raw.fit(X_train, y_train)  # NON-standardized features

print('lr_raw.intercept_        :', lr_raw.intercept_)
print('y_train.mean()           :', y_train.mean())
print('diff                     :', lr_raw.intercept_ - y_train.mean())
print()

# The general formula w0 = ybar - sum(w_i * xbar_i) must always hold, regardless of scaling
manual_w0 = y_train.mean() - np.sum(lr_raw.coef_ * X_train.mean().values)
print('w0 from the formula ybar - sum(w_i * xbar_i):', manual_w0)
print('lr_raw.intercept_ (cross-check)       :', lr_raw.intercept_)
print('diff                                   :', manual_w0 - lr_raw.intercept_)

### Ridge: the explicit solution

We have just made sure that `Latitude` and `Longitude` correlate strongly, and their VIF is well above the threshold — one of the reasons for the instability of the weights (Blocks 3, 5). Ridge regression has an explicit solution:

$$\hat\beta^{Ridge}_\lambda = (X^TX+\lambda I)^{-1}X^Ty.$$

Whence:

$$(X^TX+\lambda I)\hat\beta^{Ridge}_\lambda = X^Ty$$

**Q12.** On the same `X_train_scaled`, `y_train`:
1. Implement the formula by hand (`numpy`, without `sklearn`) for $\lambda=1.0$.
2. Compare it with `sklearn.linear_model.Ridge(alpha=1.0, fit_intercept=False)` trained on the same data.
3. Compare the `Latitude`/`Longitude` weights in the ordinary `LinearRegression` (`lr.coef_`) and in Ridge — did they change much compared with the other features?

To find the solution of the system you will need `np.linalg.solve(A, b)`, which finds the solution of the system $Ax = b.$

In [ ]:
from sklearn.linear_model import Ridge

lam = 1.0
n_features = X_train_scaled.shape[1]

# The explicit solution
beta_ridge_manual = np.linalg.solve(
    X_train_scaled.T @ X_train_scaled + lam * np.eye(n_features),
    X_train_scaled.T @ y_train
)

# sklearn
ridge_sklearn = Ridge(alpha=lam, fit_intercept=False)
ridge_sklearn.fit(X_train_scaled, y_train)

comparison = pd.DataFrame({
    'feature': X.columns,
    'beta_OLS (lr.coef_)': lr.coef_.round(4),
    'beta_ridge_manual': beta_ridge_manual.round(4),
    'beta_ridge_sklearn': ridge_sklearn.coef_.round(4),
})
comparison['diff_manual_vs_sklearn'] = (comparison['beta_ridge_manual'] - comparison['beta_ridge_sklearn']).round(6)
comparison

### Ridge at different $\lambda$: the regularization path

Let us look at the effect in dynamics — we will train Ridge on a grid of $\lambda$ and plot the `Latitude`/`Longitude` weights against $\lambda$.

**In the trainer give the $\lambda$ that produced a zeroing for Ridge.**

In [ ]:
lambdas_ridge = [0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000]

ridge_path = []
for lam in lambdas_ridge:
    r = Ridge(alpha=lam)
    r.fit(X_train_scaled, y_train)
    ridge_path.append(r.coef_)

ridge_path = np.array(ridge_path)

plt.figure(figsize=(8, 5))
plt.plot(lambdas_ridge, ridge_path[:, list(labels).index('Latitude')], marker='o', label='Latitude')
plt.plot(lambdas_ridge, ridge_path[:, list(labels).index('Longitude')], marker='o', label='Longitude')
plt.xscale('log')
plt.xlabel('$\lambda$ (log scale)')
plt.ylabel('Weight')
plt.legend()
plt.title('Ridge: the regularization path for Latitude/Longitude');

### Lasso: zeroing at a finite $\lambda$

**Q13.** Train `Lasso` on the same grid `X_train_scaled`, `y_train` for $\lambda \in \{0.0001, 0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0\}$ (`Lasso(alpha=lam, max_iter=20000)`). For every $\lambda$ compute the number of non-zero weights and separately the `Latitude`/`Longitude` weights.

Which value of $\lambda$ led to a 0 at Latitude/Longitude for at least one of the features for Lasso?

In [ ]:
from sklearn.linear_model import Lasso

lambdas_lasso = [0.0001, 0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0]

lasso_path = []
for lam in lambdas_lasso:
    l = Lasso(alpha=lam, max_iter=20000)
    l.fit(X_train_scaled, y_train)
    lasso_path.append(l.coef_)

lasso_path = np.array(lasso_path)

lasso_report = pd.DataFrame({
    'lambda': lambdas_lasso,
    'n_nonzero': (np.abs(lasso_path) > 1e-8).sum(axis=1),
    'Latitude': lasso_path[:, list(labels).index('Latitude')].round(4),
    'Longitude': lasso_path[:, list(labels).index('Longitude')].round(4),
})
lasso_report

In [ ]:
# The full picture: what happens to ALL the weights, not only Latitude/Longitude
lasso_full = pd.DataFrame(lasso_path, columns=labels)
lasso_full.insert(0, 'lambda', lambdas_lasso)
lasso_full.insert(1, 'n_nonzero', (np.abs(lasso_path) > 1e-8).sum(axis=1))
lasso_full.round(4)

**Back to the prediction from Block 5.** Look at the full table above — the picture is richer than "regularization removes correlating features":

- **`Population` goes first** (VIF $\approx 1.1$, correlates with nothing) — it is zeroed among the very first, already by $\lambda=0.005$, on its own.
- **Second, all three at once, by $\lambda=0.05$** — `AveRooms`, `AveBedrms` **and** `AveOccup`. Note: this is a mixed group — `AveRooms`/`AveBedrms` really do correlate with each other (VIF $\approx 8.1/6.8$), while `AveOccup` correlates with nothing (VIF $\approx 1.0$) — but Lasso zeroes them **simultaneously**, because by their eventual contribution to quality they are currently in the same weight class, not because they are related to one another somehow.
- **`Longitude`** goes next, by $\lambda=0.1$, while `Latitude` still holds a small weight ($-0.0104$).
- **`Latitude` and `HouseAge` are zeroed together**, by $\lambda=0.2$ — again a pair where one feature (`Latitude`) took part in a strong correlation and the other (`HouseAge`, VIF $\approx 1.24$) did not, but by contribution at this step they have levelled out.
- **`MedInc`** holds out the longest, until $\lambda=1.0$.

If the hypothesis "regularization removes precisely the correlating features" were true, the pairs `Latitude`/`Longitude` and `AveRooms`/`AveBedrms` should have been zeroed first and cleanly — whereas in fact the order of zeroing mixes correlating and non-correlating features at every step, and the very first to go was `Population`, which correlates with nothing at all.

**The answer to the general question: no, regularization is not obliged to remove only strongly correlating features.** It penalizes weights by their contribution to reducing the error relative to the price $\lambda$ — and zeroes what is cheapest to lose, not what correlates with something. Correlation is only one of the ways to make a feature "cheap": if a pair has a partner that will cover most of its contribution, the individual loss from zeroing one of them is smaller. But it is just as cheap to zero a simply weak feature that correlates with nobody — as we saw with `Population`. And a strongly correlating but informative pair (`Latitude`/`Longitude`) outlived far weaker features, because *together* it gives a real contribution to quality, even if individually the weights are unstable (Block 3, Block 5).

That is, **correlation explains the instability of the point estimate of a weight** (VIF is responsible for that, Block 5), but does not directly determine the order of zeroing in Lasso — that is governed by the joint contribution of a feature (or a group of features) to the quality of the model relative to the penalty $\lambda$.

### Bonus: L0 head-on — an exact search

L0 regularization penalizes not the magnitude of a weight but the very fact of using it: $\|\beta\|_0=\sum_j\mathbb{1}[\beta_j\neq 0]$. In the general case this is an NP-hard combinatorial problem — a search over subsets of features. But we have only 8 features, which means only $2^8=256$ subsets — we will search them **exactly** rather than approximately.

For every $\lambda$ we look for the subset of features $S$ that minimizes $SSE(S) + \lambda\cdot|S|$, where $SSE(S)$ is the sum of squared errors of an OLS trained only on the features from $S$ (on `X_train_scaled`, `y_train`).

**Q14 (starred).** Implement the exact search for $\lambda \in \{0,\ 50,\ 200,\ 500,\ 1000,\ 3000,\ 6000,\ 10000,\ 20000\}$ and print the chosen subset of features for every $\lambda$.

At which $\lambda$ did the set of features turn out to be empty (it is cheaper for the model not to use features at all)?

In [ ]:
from itertools import combinations

def sse_for_subset(cols, Xtr, ytr):
    y_centered = ytr - ytr.mean()
    if len(cols) == 0:
        return np.sum(y_centered**2)
    Xsub = Xtr[:, cols]
    beta, *_ = np.linalg.lstsq(Xsub, y_centered, rcond=None)
    pred = Xsub @ beta
    return np.sum((y_centered - pred)**2)

feat_idx = list(range(len(labels)))
lambdas_l0 = [0, 50, 200, 500, 1000, 3000, 6000, 10000, 20000]

l0_rows = []
for lam in lambdas_l0:
    best = None
    for k in range(0, len(feat_idx)+1):
        for combo in combinations(feat_idx, k):
            sse = sse_for_subset(list(combo), X_train_scaled, y_train.values)
            obj = sse + lam*k
            if best is None or obj < best[0]:
                best = (obj, combo, sse)
    obj, combo, sse = best
    l0_rows.append({'lambda': lam, 'k': len(combo), 'features': [labels[i] for i in combo], 'SSE': round(sse, 1)})

pd.DataFrame(l0_rows)

That is all, friends! We have gone the whole way: from "which feature is the most important" through "and how confident are we in that estimate at all" and "what does the model actually predict" — to what $\beta_0$ and regularization do with that uncertainty.

You have come a long way and practised a great deal. Well done!

See you in the homeworks,  \
Your course team : ) 